In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
def vgg_block(num_convs,in_channels,out_channels):  #牺牲内部定义通道的灵活性
    layers = []
    for _ in range(num_convs):
        layers.append(nn.Conv2d(in_channels,out_channels,kernel_size=3,padding=1))  #这种3,1的配置在stride = 1 的情况下尺寸不变
        layers.append(nn.ReLU())

        in_channels = out_channels
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
    return nn.Sequential(*layers) #解包，把[]容器内的东西都打开


阶段 (Stage),配置 A (VGG-11) 内容,你的 conv_arch 对应关系
Stage 1,"conv3-64, maxpool","(1, 64)"
Stage 2,"conv3-128, maxpool","(1, 128)"
Stage 3,"conv3-256, conv3-256, maxpool","(2, 256)"
Stage 4,"conv3-512, conv3-512, maxpool","(2, 512)"
Stage 5,"conv3-512, conv3-512, maxpool","(2, 512)"

In [4]:
conv_arch  = ((1,64),(1,128),(2,256),(2,512),(2,512))

In [ ]:
def vgg(conv_arch):
    conv_blks = []
    in_channels = 1
    #VGG块部分
    for (num_convs,out_channels) in conv_arch:
        conv_blks.append(vgg_block(num_convs,in_channels,out_channels))
        in_channels = out_channels

    #全连接层部分
    return nn.Sequential(
        *conv_blks,nn.Flatten(),
        nn.Linear(out_channels *7 *7,4096),nn.ReLU(),nn.Dropout(p = 0.5),
        nn.Linear(4096,4096),nn.ReLU(),nn.Dropout(0.5),
        nn.Linear(4096,10)
    )

net = vgg(conv_arch)

In [6]:
X = torch.randn(size=(1, 1, 224, 224))
for blk in net:
    X = blk(X)
    print(blk.__class__.__name__,'output shape:\t',X.shape)

Sequential output shape:	 torch.Size([1, 64, 112, 112])
Sequential output shape:	 torch.Size([1, 128, 56, 56])
Sequential output shape:	 torch.Size([1, 256, 28, 28])
Sequential output shape:	 torch.Size([1, 512, 14, 14])
Sequential output shape:	 torch.Size([1, 512, 7, 7])
Flatten output shape:	 torch.Size([1, 25088])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 4096])
ReLU output shape:	 torch.Size([1, 4096])
Dropout output shape:	 torch.Size([1, 4096])
Linear output shape:	 torch.Size([1, 10])


由于VGG参数比较多，采取减少通道的办法

pair[0]：代表卷积层的数量（深度），保持不变。
pair[1] // ratio：代表输出通道数（宽度），全部整除 4。

变化对比：
原版：(1, 64), (1, 128), (2, 256)...
瘦身版：(1, 16), (1, 32), (2, 64)...

In [ ]:
ratio = 4
small_conv_arch = [(pair[0], pair[1] // ratio) for pair in conv_arch]  #保持VGG的深度不变，但是输出通道数全部除以4来减轻计算量
net = vgg(small_conv_arch)

In [ ]:
lr, num_epochs, batch_size = 0.05, 10, 128
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=224)
d2l.train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())